In [15]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import cv2

from google.colab import drive
drive.mount('/content/drive')

os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
IMAGE_DIR   = "/content/drive/MyDrive/floodsegnet/data/Image"
MASK_DIR    = "/content/drive/MyDrive/floodsegnet/data/Mask"
METADATA    = "/content/drive/MyDrive/floodsegnet/data/metadata.csv"

metadata = pd.read_csv(METADATA)
metadata.head()

image_paths = [
    os.path.join(IMAGE_DIR, img_name)
    for img_name in metadata["Image"]
]

mask_paths = [
    os.path.join(MASK_DIR, mask_name)
    for mask_name in metadata["Mask"]
]

print(f"Images: {len(image_paths)}, Masks: {len(mask_paths)}")

Images: 290, Masks: 290


In [19]:
IMG_SIZE = 256
BATCH_SIZE = 8

In [ ]:
def load_single(img_path, mask_path):
    # load a single image
    # read, convert to rgb [opencv reads in bgr], resize, normalise to [0,1] float32

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image.astype(np.float32) / 255.0

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)

    return image, mask

# load everything into memory upfront — dataset is only 290 images, fits in RAM
all_images = []
all_masks  = []

for img_p, msk_p in zip(image_paths, mask_paths):
    img, msk = load_single(img_p, msk_p)
    all_images.append(img)
    all_masks.append(msk)

all_images = np.array(all_images)  # (290, 256, 256, 3)
all_masks  = np.array(all_masks)   # (290, 256, 256, 1)

print(f"Images: {all_images.shape}")
print(f"Masks:  {all_masks.shape}")

Images: (290, 256, 256, 3)
Masks:  (290, 256, 256, 1)


In [ ]:
# split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    all_images, all_masks, 
    test_size=0.2,
    random_state=42
)

In [22]:
# tf datasets — no map, no numpy_function, no AutoGraph issues
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [23]:
# sanity check
for images, masks in train_dataset.take(1):
    print(f"Image batch: {images.shape}")  # (8, 256, 256, 3)
    print(f"Mask batch:  {masks.shape}")   # (8, 256, 256, 1)
    print(f"Mask unique: {np.unique(masks.numpy())}")  # [0. 1.]

Image batch: (8, 256, 256, 3)
Mask batch:  (8, 256, 256, 1)
Mask unique: [0. 1.]


In [ ]:
#the fun part
model = sm.Unet(
    backbone_name="resnet34",
    encoder_weights="imagenet",
    classes=1,
    activation="sigmoid"
)

In [26]:
model.compile(
    optimizer='adam',
    loss=sm.losses.bce_dice_loss,
    metrics=[
        sm.metrics.iou_score,
        sm.metrics.f1_score
    ]
)

In [27]:
callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_f1-score",
        mode="max",
        save_best_only=True
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_f1-score",
        mode="max",
        patience=8,
        restore_best_weights=True
    )

]

In [28]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=callbacks
)

BATCH_SIZE = 8

Epoch 1/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 83s 751ms/step - f1-score: 0.6801 - iou_score: 0.5219 - loss: 0.7492 - val_f1-score: 0.0800 - val_iou_score: 0.0417 - val_loss: 4.0290
Epoch 2/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 181ms/step - f1-score: 0.7740 - iou_score: 0.6329 - loss: 0.5732 - val_f1-score: 1.0513e-06 - val_iou_score: 5.2566e-07 - val_loss: 7.3661
Epoch 3/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 11s 389ms/step - f1-score: 0.7966 - iou_score: 0.6638 - loss: 0.5216 - val_f1-score: 0.5782 - val_iou_score: 0.4090 - val_loss: 3.5449
Epoch 4/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 15s 181ms/step - f1-score: 0.8103 - iou_score: 0.6838 - loss: 0.4827 - val_f1-score: 0.4914 - val_iou_score: 0.3278 - val_loss: 2.4423
Epoch 5/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 10s 182ms/step - f1-score: 0.8225 - iou_score: 0.7012 - loss: 0.4588 - val_f1-score: 0.3835 - val_iou_score: 0.2375 - val_loss: 1.2865
Epoch 6/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 182ms/step - f1-score: 0.8104 - iou_score: 0.6833 - loss: 0.4907 - val_f1-score: 0.5767

In [29]:
model.save("/content/drive/MyDrive/floodsegnet/best_model_colab.keras")
print("Model saved!")

Model saved!
